In [5]:
# %%
import eurostat
import pandas as pd
import plotly.express as px
import pycountry

c:\Users\henri_ugzoq54\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [6]:
# %%
datasets = ["educ_uoe_lang01"]

data = {}

for ds in datasets:
    print(f"Downloading {ds}...")
    data[ds] = eurostat.get_data_df(ds, flags=False)

In [9]:
import os
print(os.getcwd())

c:\Users\henri_ugzoq54\OneDrive\TransfertDocument09-11-2025\2025_2026\Trento M1 DS\Semester 2\Data Vis


In [15]:
def clean_eurostat(df):
    # rename geo column FIRST
    for c in df.columns:
        if "geo" in c.lower():
            df = df.rename(columns={c: "geo"})

    year_cols = [c for c in df.columns if c.isdigit()]
    id_cols = [c for c in df.columns if c not in year_cols]

    df = df.melt(id_vars=id_cols, var_name="year", value_name="value")

    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df = df.dropna(subset=["value"])

    return df

In [16]:
# %%
def country_to_iso2(name):
    try:
        return pycountry.countries.lookup(name).alpha_2
    except:
        return None

mapping = {
    "Czech Republic": "Czechia",
    "Russia": "Russian Federation"
}

ef["country_clean"] = ef["country"].replace(mapping)
ef["iso2"] = ef["country_clean"].apply(country_to_iso2)

In [17]:
# %%
def iso2_to_iso3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return None

ef["iso3"] = ef["iso2"].apply(iso2_to_iso3)
ef = ef[ef["iso3"].notna()]

In [18]:
# %%
ef["fluency_rank"] = ef.groupby("year")["score"].rank(pct=True)

In [19]:
# --- ENGLISH LEARNING ---
df1 = clean_eurostat(data["educ_uoe_lang01"])

if "unit" in df1.columns:
    df1 = df1[df1["unit"] == "PC"]

if "sex" in df1.columns:
    df1 = df1[df1["sex"] == "T"]

df_final = (
    df1[df1["language"] == "ENG"]
    .groupby(["geo","year"])["value"]
    .mean()
    .reset_index()
    .rename(columns={"value": "learning"})
)

df_final = df_final[df_final["geo"].str.len() == 2]

In [20]:
print(df1["language"].unique()[:20])


['ARA' 'BUL' 'CHI' 'CZE' 'DAN' 'DUT' 'ENG' 'EST' 'FIN' 'FRE' 'GER' 'GLE'
 'GRE' 'HRV' 'HUN' 'ITA' 'JPN' 'LAV' 'LIT' 'MLT']


In [21]:
# %%
# --- CLEAN EUROSTAT (MINIMAL PROPRE) ---

def safe_filter(df):
    if "sex" in df.columns and "T" in df["sex"].unique():
        df = df[df["sex"] == "T"]
    return df

df1 = safe_filter(df1)

# garder % uniquement
if "unit" in df1.columns:
    df1 = df1[df1["unit"] == "PC"]

# nettoyer langues
df1 = df1[~df1["language"].isin(["TOTAL", "UNK", "OTH"])]

# --- VARIABLE CLÉ ---
df_final = (
    df1[df1["language"] == "ENG"]
    .groupby(["geo","year"])["value"]
    .mean()
    .reset_index()
    .rename(columns={"value": "learning"})
)

# garder pays
df_final = df_final[df_final["geo"].str.len() == 2]

In [22]:
print("AFTER FILTER ↓")

print("df1 languages:", df1["language"].unique())

print("df1 shape:", df1.shape)


AFTER FILTER ↓
df1 languages: ['ARA' 'BUL' 'CHI' 'CZE' 'DAN' 'DUT' 'ENG' 'EST' 'FIN' 'FRE' 'GER' 'GLE'
 'GRE' 'HRV' 'HUN' 'ITA' 'JPN' 'LAV' 'LIT' 'MLT' 'POL' 'POR' 'RUM' 'RUS'
 'SLO' 'SLV' 'SPA' 'SWE']
df1 shape: (49706, 7)


In [25]:
# %%
ef = pd.read_csv("data/efiepi_rankings.csv")

ef = ef.rename(columns={
    "Country": "country",
    "Year": "year",
    "Score": "score"
})

ef["year"] = pd.to_numeric(ef["year"])
ef["score"] = pd.to_numeric(ef["score"])

In [23]:
# %%
def iso2_to_iso3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return None

df_final["iso3"] = df_final["geo"].apply(iso2_to_iso3)
df_final = df_final[df_final["iso3"].notna()]

In [26]:
# %%
def country_to_iso2(name):
    try:
        return pycountry.countries.lookup(name).alpha_2
    except:
        return None

mapping = {
    "Czech Republic": "Czechia",
    "Russia": "Russian Federation"
}

ef["country_clean"] = ef["country"].replace(mapping)
ef["iso2"] = ef["country_clean"].apply(country_to_iso2)

ef["iso3"] = ef["iso2"].apply(iso2_to_iso3)
ef = ef[ef["iso3"].notna()]

In [27]:
# %%
ef["fluency_rank"] = ef.groupby("year")["score"].rank(pct=True)

In [32]:
# %%
df = df_final.merge(
    ef[["iso3","year","fluency_rank"]],
    on=["iso3","year"],
    how="left"   # ← IMPORTANT
)

# %%
# STANDARDISATION (clé)
df["learning_z"] = df.groupby("year")["learning"].transform(
    lambda x: (x - x.mean()) / x.std()
)

# GAP CORRECT
df["gap"] = df["fluency_rank"] - df["learning_z"]

In [36]:
df = df[df["fluency_rank"].notna()]
# %%
df = df.sort_values("year")

In [37]:
# %%
fig = px.choropleth(
    df,
    locations="iso3",
    color="gap",
    animation_frame="year",
    hover_name="geo",
    color_continuous_scale="RdBu",
    range_color=[-1,1],   # ← IMPORTANT
    title="Efficiency (standardized): Learning vs Fluency"
)

fig.update_geos(scope="europe")
fig.show()

In [35]:
print(df.shape)
print(df.describe())
print(df.head())
# %%
print(df[["learning","learning_z","fluency_rank","gap"]].describe())

(364, 7)
              year    learning  fluency_rank    learning_z         gap
count   364.000000  364.000000    167.000000  3.640000e+02  167.000000
mean   2018.403846   84.581809      0.623044 -1.500631e-16    0.687814
std       3.566617   15.996156      0.247343  9.833322e-01    0.969235
min    2012.000000    0.000000      0.066667 -4.218636e+00   -0.697399
25%    2015.000000   73.655000      0.421216 -6.935611e-01   -0.028094
50%    2018.000000   91.240000      0.645161  4.194100e-01    0.328519
75%    2022.000000   97.035000      0.836022  7.403938e-01    1.475965
max    2024.000000  100.000000      1.000000  1.344608e+00    2.878024
    geo  year  learning iso3  fluency_rank  learning_z  gap
1    AT  2012     98.60  AUT           NaN    1.344608  NaN
65   DE  2012     72.10  DEU           NaN   -0.401422  NaN
311  RO  2012     88.08  ROU           NaN    0.651467  NaN
78   DK  2012     66.02  DNK           NaN   -0.802021  NaN
225  LV  2012     86.40  LVA           NaN    0.5407